In [ ]:
# -*- coding: utf-8 -*-
"""
Mini-Recouvrement - Version Finale Propre
Modélisation avec split Train/Validation/Test
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configuration
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
pd.set_option('display.max_rows', None)

# Imports ML
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    recall_score, precision_score, f1_score,
    accuracy_score, classification_report,
    confusion_matrix, roc_auc_score, roc_curve
)

print("=" * 60)
print("MINI-RECOUVREMENT - MODÉLISATION FINALE")
print("=" * 60)

MINI-RECOUVREMENT - MODÉLISATION FINALE


In [ ]:
# ============================================================================
# ÉTAPE 1: CHARGEMENT ET NETTOYAGE
# ============================================================================

print("\n[1/7] Chargement des données...")

# Charger le dataset
df = pd.read_csv('data_clean.csv', low_memory=False)
print(f"   ✓ Dataset chargé: {df.shape}")

# Supprimer colonnes non nécessaires
cols_to_drop = ['purpose', 'purpose_step1', 'purpose_risk_class', 'earliest_cr_line_dt']
df = df.drop(columns=[col for col in cols_to_drop if col in df.columns])

# Supprimer les NaN dans la cible
df = df.dropna(subset=['default_risk'])
print(f"   ✓ Après nettoyage: {df.shape}")

# Gestion valeurs manquantes (ton code existant)
print("\n[2/7] Gestion des valeurs manquantes...")

# Variables de comptage → 0
count_vars = ['open_act_il', 'open_il_24m', 'open_rv_12m', 'open_rv_24m',
              'inq_fi', 'total_cu_tl', 'inq_last_12m', 'open_acc_6m', 'open_il_12m']

# Créer indicatrices avant imputation
for col in count_vars:
    if col in df.columns:
        df[f'{col}_missing'] = df[col].isnull().astype(int)

df[count_vars] = df[count_vars].fillna(0)

# Variables continues → médiane
cont_vars = ['total_bal_il', 'dti', 'all_util', 'max_bal_bc']
for col in cont_vars:
    if col in df.columns:
        df[f'{col}_missing'] = df[col].isnull().astype(int)
        df[col] = df[col].fillna(df[col].median())

# Variable catégorielle
if 'home_ownership_ordinal' in df.columns:
    df['home_ownership_ordinal'] = df['home_ownership_ordinal'].fillna(1.0)

print(f"   ✓ Valeurs manquantes restantes: {df.isnull().sum().sum()}")

# Feature Engineering (tes meilleures features)
print("\n[3/7] Feature Engineering...")

# Ratios financiers
df['payment_burden'] = np.where(
    df['annual_inc'] > 0,
    df['installment'] / df['annual_inc'],
    0
)

df['credit_utilization_ratio'] = np.where(
    df['revol_bal'] + df['installment'] > 0,
    df['revol_bal'] / (df['revol_bal'] + df['installment']),
    0
)

df['debt_stress_score'] = np.where(
    df['annual_inc'] > 0,
    df['int_rate'] * df['dti'] / df['annual_inc'],
    0
)

df['experience_vs_debt'] = df['credit_history_years'] / (df['dti'] + 1)

df['financial_health_score'] = np.where(
    df['tot_cur_bal'] + df['installment'] > 0,
    df['annual_inc'] / (df['tot_cur_bal'] + df['installment']),
    0
)

print(f"   ✓ Features créées: 5 nouveaux ratios")

# Nettoyage outliers critiques
print("\n[4/7] Nettoyage des outliers...")
df['delinq_2yrs'] = df['delinq_2yrs'].clip(upper=4)
df['annual_inc'] = df['annual_inc'].clip(upper=300000)
df['revol_bal'] = df['revol_bal'].clip(upper=100000)
print("   ✓ Outliers clippés")

# Vérification finale
print(f"\n   Dataset final: {df.shape}")
print(f"   Variables numériques: {df.select_dtypes(include=[np.number]).shape[1]}")
print(f"   Valeurs manquantes: {df.isnull().sum().sum()}")


[1/7] Chargement des données...
   ✓ Dataset chargé: (2019, 105)
   ✓ Après nettoyage: (2018, 101)

[2/7] Gestion des valeurs manquantes...
   ✓ Valeurs manquantes restantes: 0

[3/7] Feature Engineering...
   ✓ Features créées: 5 nouveaux ratios

[4/7] Nettoyage des outliers...
   ✓ Outliers clippés

   Dataset final: (2018, 119)
   Variables numériques: 119
   Valeurs manquantes: 0


In [ ]:
# ============================================================================
# ÉTAPE 2: SPLIT TRAIN / VALIDATION / TEST (70/15/15)
# ============================================================================

print("\n[5/7] Split Train/Validation/Test...")

# Séparer X et y
X = df.drop('default_risk', axis=1)
y = df['default_risk']

# Supprimer colonnes non-numériques restantes
cols_non_num = X.select_dtypes(include=['object']).columns.tolist()
if cols_non_num:
    print(f"   ⚠ Suppression colonnes non-numériques: {cols_non_num}")
    X = X.drop(columns=cols_non_num)

print(f"   Dataset X: {X.shape}")
print(f"   Distribution y: {y.value_counts().to_dict()}")

# Premier split: 70% train, 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# Deuxième split: 15% validation, 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print(f"\n   ✓ Train:      {X_train.shape[0]:,} observations ({len(X_train)/len(X)*100:.1f}%)")
print(f"   ✓ Validation: {X_val.shape[0]:,} observations ({len(X_val)/len(X)*100:.1f}%)")
print(f"   ✓ Test:       {X_test.shape[0]:,} observations ({len(X_test)/len(X)*100:.1f}%)")

# Vérifier distribution cible
print(f"\n   Distribution défauts:")
print(f"      Train:      {y_train.sum()}/{len(y_train)} = {y_train.mean()*100:.1f}%")
print(f"      Validation: {y_val.sum()}/{len(y_val)} = {y_val.mean()*100:.1f}%")
print(f"      Test:       {y_test.sum()}/{len(y_test)} = {y_test.mean()*100:.1f}%")

# Standardisation (pour Logistic Regression)
print("\n[6/7] Standardisation des données...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("   ✓ Standardisation appliquée (mean=0, std=1)")
print(f"   ✓ X_train_scaled: {X_train_scaled.shape}")


[5/7] Split Train/Validation/Test...
   Dataset X: (2018, 118)
   Distribution y: {0.0: 1676, 1.0: 342}

   ✓ Train:      1,412 observations (70.0%)
   ✓ Validation: 303 observations (15.0%)
   ✓ Test:       303 observations (15.0%)

   Distribution défauts:
      Train:      239.0/1412 = 16.9%
      Validation: 52.0/303 = 17.2%
      Test:       51.0/303 = 16.8%

[6/7] Standardisation des données...
   ✓ Standardisation appliquée (mean=0, std=1)
   ✓ X_train_scaled: (1412, 118)


In [ ]:
# ============================================================================
# ÉTAPE 3: ENTRAÎNEMENT DES MODÈLES
# ============================================================================

print("\n[7/7] Entraînement des modèles...")
print("=" * 60)

# Dictionnaire pour stocker les résultats
resultats_validation = {}

# ============================================================================
# MODÈLE 1: RANDOM FOREST
# ============================================================================

print("\n[1/3] Random Forest...")
rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=12,
    min_samples_split=2,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

print("   Entraînement en cours...", end=" ")
rf.fit(X_train, y_train)
print("✓")

# Prédictions sur validation
y_val_pred_rf = rf.predict(X_val)
y_val_proba_rf = rf.predict_proba(X_val)[:, 1]

# Métriques de base
resultats_validation['Random Forest'] = {
    'model': rf,
    'X_data': X_val,  # Pas de scaling pour RF
    'y_proba': y_val_proba_rf,
    'metrics_default': {
        'recall': recall_score(y_val, y_val_pred_rf),
        'precision': precision_score(y_val, y_val_pred_rf),
        'f1': f1_score(y_val, y_val_pred_rf),
        'accuracy': accuracy_score(y_val, y_val_pred_rf)
    }
}

print(f"   Recall:    {resultats_validation['Random Forest']['metrics_default']['recall']:.3f}")
print(f"   Precision: {resultats_validation['Random Forest']['metrics_default']['precision']:.3f}")
print(f"   F1-Score:  {resultats_validation['Random Forest']['metrics_default']['f1']:.3f}")

# ============================================================================
# MODÈLE 2: LOGISTIC REGRESSION
# ============================================================================

print("\n[2/3] Logistic Regression...")
lr = LogisticRegression(
    C=100.0,
    penalty='l1',
    solver='liblinear',
    class_weight='balanced',
    random_state=42,
    max_iter=2000
)

print("   Entraînement en cours...", end=" ")
lr.fit(X_train_scaled, y_train)
print("✓")

# Prédictions sur validation
y_val_pred_lr = lr.predict(X_val_scaled)
y_val_proba_lr = lr.predict_proba(X_val_scaled)[:, 1]

resultats_validation['Logistic Regression'] = {
    'model': lr,
    'X_data': X_val_scaled,  # Avec scaling pour LR
    'y_proba': y_val_proba_lr,
    'metrics_default': {
        'recall': recall_score(y_val, y_val_pred_lr),
        'precision': precision_score(y_val, y_val_pred_lr),
        'f1': f1_score(y_val, y_val_pred_lr),
        'accuracy': accuracy_score(y_val, y_val_pred_lr)
    }
}

print(f"   Recall:    {resultats_validation['Logistic Regression']['metrics_default']['recall']:.3f}")
print(f"   Precision: {resultats_validation['Logistic Regression']['metrics_default']['precision']:.3f}")
print(f"   F1-Score:  {resultats_validation['Logistic Regression']['metrics_default']['f1']:.3f}")

# ============================================================================
# MODÈLE 3: CATBOOST
# ============================================================================

print("\n[3/3] CatBoost...")

try:
    from catboost import CatBoostClassifier

    cat = CatBoostClassifier(
        iterations=200,
        depth=6,
        learning_rate=0.1,
        class_weights=[1, 3],
        random_seed=42,
        verbose=False
    )

    print("   Entraînement en cours...", end=" ")
    cat.fit(X_train, y_train)
    print("✓")

    # Prédictions sur validation
    y_val_pred_cat = cat.predict(X_val)
    y_val_proba_cat = cat.predict_proba(X_val)[:, 1]

    resultats_validation['CatBoost'] = {
        'model': cat,
        'X_data': X_val,  # Pas de scaling pour CatBoost
        'y_proba': y_val_proba_cat,
        'metrics_default': {
            'recall': recall_score(y_val, y_val_pred_cat),
            'precision': precision_score(y_val, y_val_pred_cat),
            'f1': f1_score(y_val, y_val_pred_cat),
            'accuracy': accuracy_score(y_val, y_val_pred_cat)
        }
    }

    print(f"   Recall:    {resultats_validation['CatBoost']['metrics_default']['recall']:.3f}")
    print(f"   Precision: {resultats_validation['CatBoost']['metrics_default']['precision']:.3f}")
    print(f"   F1-Score:  {resultats_validation['CatBoost']['metrics_default']['f1']:.3f}")

except ImportError:
    print("   ⚠ CatBoost non installé, passage au suivant")

print("\n" + "=" * 60)
print("✓ ENTRAÎNEMENT TERMINÉ")
print("=" * 60)


[7/7] Entraînement des modèles...

[1/3] Random Forest...
   Entraînement en cours... ✓
   Recall:    0.058
   Precision: 0.500
   F1-Score:  0.103

[2/3] Logistic Regression...
   Entraînement en cours... ✓
   Recall:    0.692
   Precision: 0.409
   F1-Score:  0.514

[3/3] CatBoost...
   ⚠ CatBoost non installé, passage au suivant

✓ ENTRAÎNEMENT TERMINÉ


In [ ]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 8.0 MB/s eta 0:00:00


In [ ]:

print("\n[3/3] CatBoost...")

try:
    from catboost import CatBoostClassifier

    cat = CatBoostClassifier(
        iterations=200,
        depth=6,
        learning_rate=0.1,
        class_weights=[1, 3],
        random_seed=42,
        verbose=False
    )

    print("   Entraînement en cours...", end=" ")
    cat.fit(X_train, y_train)
    print("✓")

    # Prédictions sur validation
    y_val_pred_cat = cat.predict(X_val)
    y_val_proba_cat = cat.predict_proba(X_val)[:, 1]

    resultats_validation['CatBoost'] = {
        'model': cat,
        'X_data': X_val,  # Pas de scaling pour CatBoost
        'y_proba': y_val_proba_cat,
        'metrics_default': {
            'recall': recall_score(y_val, y_val_pred_cat),
            'precision': precision_score(y_val, y_val_pred_cat),
            'f1': f1_score(y_val, y_val_pred_cat),
            'accuracy': accuracy_score(y_val, y_val_pred_cat)
        }
    }

    print(f"   Recall:    {resultats_validation['CatBoost']['metrics_default']['recall']:.3f}")
    print(f"   Precision: {resultats_validation['CatBoost']['metrics_default']['precision']:.3f}")
    print(f"   F1-Score:  {resultats_validation['CatBoost']['metrics_default']['f1']:.3f}")

except ImportError:
    print("   ⚠ CatBoost non installé, passage au suivant")

print("\n" + "=" * 60)
print("✓ ENTRAÎNEMENT TERMINÉ")
print("=" * 60)


[3/3] CatBoost...
   Entraînement en cours... ✓
   Recall:    0.288
   Precision: 0.682
   F1-Score:  0.405

✓ ENTRAÎNEMENT TERMINÉ


In [ ]:
# ============================================================================
# ÉTAPE 4: OPTIMISATION DES SEUILS SUR VALIDATION
# ============================================================================

print("\n" + "=" * 60)
print("OPTIMISATION DES SEUILS SUR ENSEMBLE DE VALIDATION")
print("=" * 60)

# Seuils à tester
seuils = [0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6]

# Stocker tous les résultats
tous_resultats = []

for nom_modele, data in resultats_validation.items():
    print(f"\n{nom_modele}")
    print("-" * 60)
    print(f"{'Seuil':<8} {'Recall':<10} {'Precision':<12} {'F1-Score':<10} {'Accuracy':<10}")
    print("-" * 60)

    y_proba = data['y_proba']

    for seuil in seuils:
        # Prédictions avec ce seuil
        y_pred = (y_proba >= seuil).astype(int)

        # Calculer métriques
        recall = recall_score(y_val, y_pred)
        precision = precision_score(y_val, y_pred)
        f1 = f1_score(y_val, y_pred)
        accuracy = accuracy_score(y_val, y_pred)

        # Afficher
        print(f"{seuil:<8.2f} {recall:<10.3f} {precision:<12.3f} {f1:<10.3f} {accuracy:<10.3f}")

        # Stocker
        tous_resultats.append({
            'modele': nom_modele,
            'seuil': seuil,
            'recall': recall,
            'precision': precision,
            'f1': f1,
            'accuracy': accuracy
        })

# ============================================================================
# SÉLECTION DU MEILLEUR MODÈLE + SEUIL
# ============================================================================

print("\n" + "=" * 60)
print("SÉLECTION DU MEILLEUR MODÈLE")
print("=" * 60)

# Convertir en DataFrame pour analyse facile
df_resultats = pd.DataFrame(tous_resultats)

# Critère 1: F1-Score maximal
meilleur_f1 = df_resultats.loc[df_resultats['f1'].idxmax()]

print("\n[CRITÈRE: F1-Score maximal]")
print(f"   Modèle:    {meilleur_f1['modele']}")
print(f"   Seuil:     {meilleur_f1['seuil']:.2f}")
print(f"   Recall:    {meilleur_f1['recall']:.3f} ({meilleur_f1['recall']*100:.1f}%)")
print(f"   Precision: {meilleur_f1['precision']:.3f} ({meilleur_f1['precision']*100:.1f}%)")
print(f"   F1-Score:  {meilleur_f1['f1']:.3f}")
print(f"   Accuracy:  {meilleur_f1['accuracy']:.3f}")

# Critère 2: Meilleur équilibre (recall > 0.60 ET precision max)
df_filtre = df_resultats[df_resultats['recall'] >= 0.60]
if len(df_filtre) > 0:
    meilleur_equilibre = df_filtre.loc[df_filtre['precision'].idxmax()]

    print("\n[CRITÈRE: Meilleur équilibre (Recall≥60% + Precision max)]")
    print(f"   Modèle:    {meilleur_equilibre['modele']}")
    print(f"   Seuil:     {meilleur_equilibre['seuil']:.2f}")
    print(f"   Recall:    {meilleur_equilibre['recall']:.3f} ({meilleur_equilibre['recall']*100:.1f}%)")
    print(f"   Precision: {meilleur_equilibre['precision']:.3f} ({meilleur_equilibre['precision']*100:.1f}%)")
    print(f"   F1-Score:  {meilleur_equilibre['f1']:.3f}")
    print(f"   Accuracy:  {meilleur_equilibre['accuracy']:.3f}")

# Critère 3: Recall maximal (detection maximale)
meilleur_recall = df_resultats.loc[df_resultats['recall'].idxmax()]

print("\n[CRITÈRE: Recall maximal (détection maximale)]")
print(f"   Modèle:    {meilleur_recall['modele']}")
print(f"   Seuil:     {meilleur_recall['seuil']:.2f}")
print(f"   Recall:    {meilleur_recall['recall']:.3f} ({meilleur_recall['recall']*100:.1f}%)")
print(f"   Precision: {meilleur_recall['precision']:.3f} ({meilleur_recall['precision']*100:.1f}%)")
print(f"   F1-Score:  {meilleur_recall['f1']:.3f}")
print(f"   Accuracy:  {meilleur_recall['accuracy']:.3f}")

# ============================================================================
# CHOIX FINAL
# ============================================================================

print("\n" + "=" * 60)
print("DÉCISION FINALE")
print("=" * 60)

# On privilégie F1-score pour l'équilibre
config_finale = meilleur_f1.to_dict()

print(f"\n✓ Modèle retenu: {config_finale['modele']}")
print(f"✓ Seuil optimal: {config_finale['seuil']:.2f}")
print(f"\nPerformances sur VALIDATION:")
print(f"   - Recall:    {config_finale['recall']:.3f} ({config_finale['recall']*100:.1f}%)")
print(f"   - Precision: {config_finale['precision']:.3f} ({config_finale['precision']*100:.1f}%)")
print(f"   - F1-Score:  {config_finale['f1']:.3f}")
print(f"   - Accuracy:  {config_finale['accuracy']:.3f}")

print("\n" + "=" * 60)


OPTIMISATION DES SEUILS SUR ENSEMBLE DE VALIDATION

Random Forest
------------------------------------------------------------
Seuil    Recall     Precision    F1-Score   Accuracy  
------------------------------------------------------------
0.20     0.947      0.260        0.408      0.440     
0.25     0.881      0.289        0.435      0.534     
0.30     0.792      0.325        0.460      0.622     
0.35     0.715      0.366        0.485      0.690     
0.40     0.626      0.409        0.495      0.740     
0.45     0.541      0.458        0.496      0.776     
0.50     0.458      0.521        0.487      0.804     
0.55     0.388      0.584        0.466      0.819     
0.60     0.314      0.633        0.420      0.823     

Logistic Regression
------------------------------------------------------------
Seuil    Recall     Precision    F1-Score   Accuracy  
------------------------------------------------------------
0.20     0.964      0.241        0.385      0.374     
0.25    

In [ ]:
# ============================================================================
# FEATURE SELECTION AVANCÉE
# ============================================================================

print("\n" + "=" * 60)
print("FEATURE SELECTION AVANCÉE")
print("=" * 60)

from sklearn.feature_selection import RFE, SelectFromModel
from sklearn.ensemble import RandomForestClassifier

print(f"\n[Avant] Nombre de features: {X_train.shape[1]}")

# ----------------------------------------------------------------------------
# Méthode 1: Feature Importance (Random Forest)
# ----------------------------------------------------------------------------

print("\n[1/3] Feature Importance avec Random Forest...")

# Entraîner RF rapide pour importance
rf_selector = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf_selector.fit(X_train, y_train)

# Récupérer importance
feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': rf_selector.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 15 features les plus importantes:")
print(feature_importance.head(15))

# Calculer importance cumulée
feature_importance['cumsum'] = feature_importance['importance'].cumsum()

# Garder features qui représentent 95% de l'importance
seuil_cumsum = 0.95
n_features_95 = (feature_importance['cumsum'] <= seuil_cumsum).sum()

print(f"\n✓ {n_features_95} features représentent {seuil_cumsum*100}% de l'importance totale")

# ----------------------------------------------------------------------------
# Méthode 2: RFE (Recursive Feature Elimination)
# ----------------------------------------------------------------------------

print("\n[2/3] Recursive Feature Elimination (RFE)...")

from sklearn.linear_model import LogisticRegression

# Estimateur pour RFE (LR rapide)
lr_rfe = LogisticRegression(
    penalty='l1',
    solver='liblinear',
    class_weight='balanced',
    random_state=42,
    max_iter=1000
)

# RFE pour sélectionner top 50 features
rfe = RFE(
    estimator=lr_rfe,
    n_features_to_select=50,
    step=10
)

print(f"   Sélection de 50 features parmi {X_train.shape[1]}...", end=" ")
rfe.fit(X_train_scaled, y_train)
print("✓")

# Features sélectionnées par RFE
rfe_features = X_train.columns[rfe.support_].tolist()
print(f"   ✓ RFE a sélectionné {len(rfe_features)} features")

# ----------------------------------------------------------------------------
# Méthode 3: Combinaison (Intersection ou Union)
# ----------------------------------------------------------------------------

print("\n[3/3] Combinaison des méthodes...")

# Features du top importance
importance_features = feature_importance.head(n_features_95)['feature'].tolist()

# Intersection: features présentes dans les 2 méthodes
features_intersection = list(set(importance_features) & set(rfe_features))

# Union: features présentes dans au moins 1 méthode
features_union = list(set(importance_features) | set(rfe_features))

print(f"\n   Importance seule:     {len(importance_features)} features")
print(f"   RFE seul:             {len(rfe_features)} features")
print(f"   Intersection (ET):    {len(features_intersection)} features")
print(f"   Union (OU):           {len(features_union)} features")

# ----------------------------------------------------------------------------
# DÉCISION: Quelle stratégie adopter?
# ----------------------------------------------------------------------------

print("\n" + "=" * 60)
print("STRATÉGIE DE SÉLECTION")
print("=" * 60)

# On va tester les 3 approches et garder la meilleure

strategies = {
    'Importance_95': importance_features,
    'RFE_50': rfe_features,
    'Intersection': features_intersection,
    'Union': features_union
}

resultats_strategies = []

print("\nTest rapide des stratégies sur Logistic Regression...")

for nom_strategie, features_selected in strategies.items():
    # Créer les datasets filtrés
    X_train_sel = X_train[features_selected]
    X_val_sel = X_val[features_selected]

    # Scaling
    scaler_temp = StandardScaler()
    X_train_sel_scaled = scaler_temp.fit_transform(X_train_sel)
    X_val_sel_scaled = scaler_temp.transform(X_val_sel)

    # Modèle rapide
    lr_temp = LogisticRegression(
        C=100.0,
        penalty='l1',
        solver='liblinear',
        class_weight='balanced',
        random_state=42,
        max_iter=2000
    )

    lr_temp.fit(X_train_sel_scaled, y_train)
    y_pred_temp = lr_temp.predict(X_val_sel_scaled)

    # Métriques
    f1_temp = f1_score(y_val, y_pred_temp)
    recall_temp = recall_score(y_val, y_pred_temp)
    precision_temp = precision_score(y_val, y_pred_temp)

    resultats_strategies.append({
        'strategie': nom_strategie,
        'n_features': len(features_selected),
        'f1': f1_temp,
        'recall': recall_temp,
        'precision': precision_temp
    })

    print(f"   {nom_strategie:20s} ({len(features_selected):3d} features): F1={f1_temp:.3f}, Recall={recall_temp:.3f}, Precision={precision_temp:.3f}")

# Sélectionner meilleure stratégie
df_strategies = pd.DataFrame(resultats_strategies)
meilleure_strategie = df_strategies.loc[df_strategies['f1'].idxmax()]

print("\n" + "=" * 60)
print("✓ MEILLEURE STRATÉGIE")
print("=" * 60)
print(f"\nStratégie:  {meilleure_strategie['strategie']}")
print(f"Features:   {int(meilleure_strategie['n_features'])}")
print(f"F1-Score:   {meilleure_strategie['f1']:.3f}")
print(f"Recall:     {meilleure_strategie['recall']:.3f}")
print(f"Precision:  {meilleure_strategie['precision']:.3f}")

# ----------------------------------------------------------------------------
# APPLIQUER LA SÉLECTION
# ----------------------------------------------------------------------------

print("\n[Application] Réduction du dataset aux features sélectionnées...")

features_finales = strategies[meilleure_strategie['strategie']]

# Réduire les datasets
X_train = X_train[features_finales]
X_val = X_val[features_finales]
X_test = X_test[features_finales]

# Re-scaler avec features réduites
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(f"\n✓ Dataset réduit:")
print(f"   Avant: 118 features")
print(f"   Après: {X_train.shape[1]} features")
print(f"   Réduction: {(1 - X_train.shape[1]/118)*100:.1f}%")

# Sauvegarder la liste des features sélectionnées
with open('features_selectionnees.txt', 'w') as f:
    f.write(f"Stratégie: {meilleure_strategie['strategie']}\n")
    f.write(f"Nombre: {len(features_finales)}\n\n")
    f.write("Features sélectionnées:\n")
    for feat in features_finales:
        f.write(f"  - {feat}\n")

print("\n✓ Liste sauvegardée dans 'features_selectionnees.txt'")
print("=" * 60)


FEATURE SELECTION AVANCÉE

[Avant] Nombre de features: 118

[1/3] Feature Importance avec Random Forest...

Top 15 features les plus importantes:
                          feature  importance
3                        int_rate    0.119705
99   debt_settlement_flag_encoded    0.107688
90                   term_encoded    0.062689
115             debt_stress_score    0.052687
113                payment_burden    0.021153
38           acc_open_past_24mths    0.020061
116            experience_vs_debt    0.019866
9                 fico_range_high    0.019203
6                             dti    0.018667
40                 bc_open_to_buy    0.018318
8                  fico_range_low    0.018283
39                    avg_cur_bal    0.016593
71                tot_hi_cred_lim    0.015821
114      credit_utilization_ratio    0.015563
4                     installment    0.015118

✓ 64 features représentent 95.0% de l'importance totale

[2/3] Recursive Feature Elimination (RFE)...
   Sélection d

In [ ]:
# ============================================================================
# HYPERPARAMETER TUNING AVANCÉ AVEC OPTUNA
# ============================================================================

print("\n" + "=" * 60)
print("HYPERPARAMETER TUNING AVEC OPTUNA")
print("=" * 60)

import optuna
from optuna.samplers import TPESampler

# Supprimer les logs verbeux d'Optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ----------------------------------------------------------------------------
# TUNING 1: LOGISTIC REGRESSION
# ----------------------------------------------------------------------------

print("\n[1/3] Optimisation Logistic Regression...")
print("   (Ceci peut prendre 15-20 minutes)")

def objective_lr(trial):
    """Fonction objectif pour Logistic Regression"""

    # Hyperparamètres à optimiser
    C = trial.suggest_float('C', 0.001, 1000.0, log=True)
    penalty = trial.suggest_categorical('penalty', ['l1', 'l2'])
    solver = 'liblinear' if penalty == 'l1' else 'lbfgs'

    # Class weight
    class_weight_type = trial.suggest_categorical('class_weight_type', ['balanced', 'manual'])
    if class_weight_type == 'manual':
        pos_weight = trial.suggest_int('pos_weight', 2, 10)
        class_weight = {0: 1, 1: pos_weight}
    else:
        class_weight = 'balanced'

    # Modèle
    lr = LogisticRegression(
        C=C,
        penalty=penalty,
        solver=solver,
        class_weight=class_weight,
        random_state=42,
        max_iter=2000
    )

    # Entraîner sur train
    lr.fit(X_train_scaled, y_train)

    # Prédire sur validation
    y_pred = lr.predict(X_val_scaled)

    # Objectif: maximiser F1-score
    f1 = f1_score(y_val, y_pred)

    return f1

# Créer l'étude Optuna
study_lr = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=42)
)

# Optimisation (100 essais)
print("   Optimisation en cours (100 essais)...", end=" ")
study_lr.optimize(objective_lr, n_trials=100, show_progress_bar=False)
print("✓")

# Meilleurs paramètres
best_params_lr = study_lr.best_params
best_f1_lr = study_lr.best_value

print(f"\n   ✓ Meilleur F1-Score: {best_f1_lr:.4f}")
print(f"   ✓ Meilleurs paramètres:")
for param, value in best_params_lr.items():
    print(f"      - {param}: {value}")

# Entraîner modèle avec meilleurs params
if best_params_lr['class_weight_type'] == 'manual':
    class_weight_final = {0: 1, 1: best_params_lr['pos_weight']}
else:
    class_weight_final = 'balanced'

solver_final = 'liblinear' if best_params_lr['penalty'] == 'l1' else 'lbfgs'

lr_optimal = LogisticRegression(
    C=best_params_lr['C'],
    penalty=best_params_lr['penalty'],
    solver=solver_final,
    class_weight=class_weight_final,
    random_state=42,
    max_iter=2000
)

lr_optimal.fit(X_train_scaled, y_train)

# ----------------------------------------------------------------------------
# TUNING 2: RANDOM FOREST
# ----------------------------------------------------------------------------

print("\n[2/3] Optimisation Random Forest...")
print("   (Ceci peut prendre 30-40 minutes)")

def objective_rf(trial):
    """Fonction objectif pour Random Forest"""

    # Hyperparamètres
    n_estimators = trial.suggest_int('n_estimators', 100, 1000, step=100)
    max_depth = trial.suggest_int('max_depth', 5, 30)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 20)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
    max_features = trial.suggest_categorical('max_features', ['sqrt', 'log2'])

    # Class weight
    class_weight_type = trial.suggest_categorical('class_weight_type', ['balanced', 'balanced_subsample'])

    # Modèle
    rf = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        class_weight=class_weight_type,
        random_state=42,
        n_jobs=-1
    )

    # Entraîner
    rf.fit(X_train, y_train)

    # Prédire
    y_pred = rf.predict(X_val)

    # F1-score
    f1 = f1_score(y_val, y_pred)

    return f1

# Optimisation RF
study_rf = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=42)
)

print("   Optimisation en cours (50 essais)...", end=" ")
study_rf.optimize(objective_rf, n_trials=50, show_progress_bar=False)
print("✓")

best_params_rf = study_rf.best_params
best_f1_rf = study_rf.best_value

print(f"\n   ✓ Meilleur F1-Score: {best_f1_rf:.4f}")
print(f"   ✓ Meilleurs paramètres:")
for param, value in best_params_rf.items():
    print(f"      - {param}: {value}")

# Entraîner RF optimal
rf_optimal = RandomForestClassifier(
    n_estimators=best_params_rf['n_estimators'],
    max_depth=best_params_rf['max_depth'],
    min_samples_split=best_params_rf['min_samples_split'],
    min_samples_leaf=best_params_rf['min_samples_leaf'],
    max_features=best_params_rf['max_features'],
    class_weight=best_params_rf['class_weight_type'],
    random_state=42,
    n_jobs=-1
)

rf_optimal.fit(X_train, y_train)

# ----------------------------------------------------------------------------
# TUNING 3: CATBOOST
# ----------------------------------------------------------------------------

print("\n[3/3] Optimisation CatBoost...")
print("   (Ceci peut prendre 30-40 minutes)")

def objective_cat(trial):
    """Fonction objectif pour CatBoost"""

    # Hyperparamètres
    iterations = trial.suggest_int('iterations', 100, 500, step=50)
    depth = trial.suggest_int('depth', 4, 10)
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
    l2_leaf_reg = trial.suggest_float('l2_leaf_reg', 1, 10)

    # Class weight
    scale_pos_weight = trial.suggest_float('scale_pos_weight', 1, 10)

    # Modèle
    cat = CatBoostClassifier(
        iterations=iterations,
        depth=depth,
        learning_rate=learning_rate,
        l2_leaf_reg=l2_leaf_reg,
        scale_pos_weight=scale_pos_weight,
        random_seed=42,
        verbose=False
    )

    # Entraîner
    cat.fit(X_train, y_train)

    # Prédire
    y_pred = cat.predict(X_val)

    # F1-score
    f1 = f1_score(y_val, y_pred)

    return f1

study_cat = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=42)
)

print("   Optimisation en cours (50 essais)...", end=" ")
study_cat.optimize(objective_cat, n_trials=50, show_progress_bar=False)
print("✓")

best_params_cat = study_cat.best_params
best_f1_cat = study_cat.best_value

print(f"\n   ✓ Meilleur F1-Score: {best_f1_cat:.4f}")
print(f"   ✓ Meilleurs paramètres:")
for param, value in best_params_cat.items():
    print(f"      - {param}: {value}")

# Entraîner CatBoost optimal
cat_optimal = CatBoostClassifier(
    iterations=best_params_cat['iterations'],
    depth=best_params_cat['depth'],
    learning_rate=best_params_cat['learning_rate'],
    l2_leaf_reg=best_params_cat['l2_leaf_reg'],
    scale_pos_weight=best_params_cat['scale_pos_weight'],
    random_seed=42,
    verbose=False
)

cat_optimal.fit(X_train, y_train)

# ----------------------------------------------------------------------------
# COMPARAISON FINALE
# ----------------------------------------------------------------------------

print("\n" + "=" * 60)
print("COMPARAISON DES MODÈLES OPTIMISÉS")
print("=" * 60)

modeles_optimaux = {
    'Logistic Regression': (lr_optimal, X_val_scaled),
    'Random Forest': (rf_optimal, X_val),
    'CatBoost': (cat_optimal, X_val)
}

resultats_optuna = []

for nom, (model, X_data) in modeles_optimaux.items():
    y_pred = model.predict(X_data)
    y_proba = model.predict_proba(X_data)[:, 1]

    recall = recall_score(y_val, y_pred)
    precision = precision_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)

    resultats_optuna.append({
        'modele': nom,
        'model_object': model,
        'X_data': X_data,
        'y_proba': y_proba,
        'recall': recall,
        'precision': precision,
        'f1': f1
    })

    print(f"\n{nom}:")
    print(f"   Recall:    {recall:.3f} ({recall*100:.1f}%)")
    print(f"   Precision: {precision:.3f} ({precision*100:.1f}%)")
    print(f"   F1-Score:  {f1:.3f}")

# Meilleur modèle
df_optuna = pd.DataFrame(resultats_optuna)
best_model_optuna = df_optuna.loc[df_optuna['f1'].idxmax()]

print("\n" + "=" * 60)
print("✓ MEILLEUR MODÈLE APRÈS TUNING")
print("=" * 60)
print(f"\nModèle:    {best_model_optuna['modele']}")
print(f"Recall:    {best_model_optuna['recall']:.3f} ({best_model_optuna['recall']*100:.1f}%)")
print(f"Precision: {best_model_optuna['precision']:.3f} ({best_model_optuna['precision']*100:.1f}%)")
print(f"F1-Score:  {best_model_optuna['f1']:.3f}")

print("\n" + "=" * 60)


HYPERPARAMETER TUNING AVEC OPTUNA

[1/3] Optimisation Logistic Regression...
   (Ceci peut prendre 15-20 minutes)
   Optimisation en cours (100 essais)... ✓

   ✓ Meilleur F1-Score: 0.5110
   ✓ Meilleurs paramètres:
      - C: 7.547315319074789
      - penalty: l1
      - class_weight_type: balanced

[2/3] Optimisation Random Forest...
   (Ceci peut prendre 30-40 minutes)
   Optimisation en cours (50 essais)... ✓

   ✓ Meilleur F1-Score: 0.5116
   ✓ Meilleurs paramètres:
      - n_estimators: 700
      - max_depth: 11
      - min_samples_split: 14
      - min_samples_leaf: 9
      - max_features: sqrt
      - class_weight_type: balanced_subsample

[3/3] Optimisation CatBoost...
   (Ceci peut prendre 30-40 minutes)
   Optimisation en cours (50 essais)... ✓

   ✓ Meilleur F1-Score: 0.5096
   ✓ Meilleurs paramètres:
      - iterations: 150
      - depth: 5
      - learning_rate: 0.0689531858171617
      - l2_leaf_reg: 8.38048621574697
      - scale_pos_weight: 2.964394344676627

COMPARAI

In [ ]:
# ============================================================================
# OPTIMISATION DES SEUILS SUR LES MODÈLES OPTIMAUX
# ============================================================================

print("\n" + "=" * 60)
print("OPTIMISATION DES SEUILS (MODÈLES PRÉ-TUNING)")
print("=" * 60)

seuils = [0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6]

# Use the models trained before Optuna
# These are stored in the resultats_validation dictionary from cell x9QEe3KiY3lS
modeles_pre_tuning = {
    'Logistic Regression': (resultats_validation['Logistic Regression']['model'], resultats_validation['Logistic Regression']['X_data']),
    'Random Forest': (resultats_validation['Random Forest']['model'], resultats_validation['Random Forest']['X_data']),
    # Check if CatBoost was successfully trained (it wasn't in the original execution output shown)
    # If it was, include it here. Otherwise, exclude it.
    # Assuming it was trained successfully after the pip install
    'CatBoost': (resultats_validation['CatBoost']['model'], resultats_validation['CatBoost']['X_data'])

}

tous_resultats_seuils = []

for nom_modele, (model, X_data) in modeles_pre_tuning.items():
    print(f"\n{nom_modele}")
    print("-" * 60)
    print(f"{'Seuil':<8} {'Recall':<10} {'Precision':<12} {'F1-Score':<10}")
    print("-" * 60)

    y_proba = model.predict_proba(X_data)[:, 1]

    for seuil in seuils:
        y_pred = (y_proba >= seuil).astype(int)

        recall = recall_score(y_val, y_pred)
        precision = precision_score(y_val, y_pred)
        f1 = f1_score(y_val, y_pred)

        print(f"{seuil:<8.2f} {recall:<10.3f} {precision:<12.3f} {f1:<10.3f}")

        tous_resultats_seuils.append({
            'modele': nom_modele,
            'seuil': seuil,
            'recall': recall,
            'precision': precision,
            'f1': f1
        })

# Sélection meilleur config
df_seuils = pd.DataFrame(tous_resultats_seuils)
meilleur_config = df_seuils.loc[df_seuils['f1'].idxmax()]

print("\n" + "=" * 60)
print("✓ CONFIGURATION OPTIMALE FINALE")
print("=" * 60)
print(f"\nModèle:    {meilleur_config['modele']}")
print(f"Seuil:     {meilleur_config['seuil']:.2f}")
print(f"Recall:    {meilleur_config['recall']:.3f} ({meilleur_config['recall']*100:.1f}%)")
print(f"Precision: {meilleur_config['precision']:.3f} ({meilleur_config['precision']*100:.1f}%)")
print(f"F1-Score:  {meilleur_config['f1']:.3f}")

# Sauvegarder config
config_finale = {
    'modele': meilleur_config['modele'],
    'seuil': float(meilleur_config['seuil']),
    'recall': float(meilleur_config['recall']),
    'precision': float(meilleur_config['precision']),
    'f1': float(meilleur_config['f1'])
}


OPTIMISATION DES SEUILS (MODÈLES PRÉ-TUNING)

Logistic Regression
------------------------------------------------------------
Seuil    Recall     Precision    F1-Score  
------------------------------------------------------------
0.20     0.885      0.245        0.383     
0.25     0.865      0.263        0.404     
0.30     0.846      0.301        0.444     
0.35     0.769      0.331        0.462     
0.40     0.750      0.355        0.481     
0.45     0.750      0.402        0.523     
0.50     0.692      0.409        0.514     
0.55     0.635      0.418        0.504     
0.60     0.577      0.429        0.492     

Random Forest
------------------------------------------------------------
Seuil    Recall     Precision    F1-Score  
------------------------------------------------------------
0.20     0.769      0.301        0.432     
0.25     0.692      0.383        0.493     
0.30     0.577      0.448        0.504     
0.35     0.385      0.435        0.408     
0.40     0.212

In [ ]:
# ============================================================================
# ÉVALUATION FINALE SUR ENSEMBLE DE TEST
# ============================================================================

print("\n" + "=" * 60)
print("ÉVALUATION FINALE SUR ENSEMBLE DE TEST")
print("=" * 60)

# Retrieve the best model and threshold from the previous step (BY9ckvL8nNR2)
# These are based on the pre-tuned models and threshold optimization on validation set.
best_model_name = meilleur_config['modele']
optimal_threshold = meilleur_config['seuil']

# Retrieve the appropriate model object from the pre-tuned models
# These models are stored in the resultats_validation dictionary from cell x9QEe3KiY3lS
# Note: X_val_scaled was used for LR, X_val for RF and CatBoost in resultats_validation
modeles_pre_tuning_objects = {
    'Logistic Regression': resultats_validation['Logistic Regression']['model'],
    'Random Forest': resultats_validation['Random Forest']['model'],
    'CatBoost': resultats_validation['CatBoost']['model']
}

model_final = modeles_pre_tuning_objects[best_model_name]

# Select the correct test set based on the chosen model's requirements (scaled or not)
if best_model_name == 'Logistic Regression':
    X_test_final = X_test_scaled
else:
    X_test_final = X_test # RF and CatBoost used non-scaled data in the validation step

# Prédictions avec seuil optimal
y_proba_test = model_final.predict_proba(X_test_final)[:, 1]
y_pred_test = (y_proba_test >= optimal_threshold).astype(int)

# Métriques
recall_test = recall_score(y_test, y_pred_test)
precision_test = precision_score(y_test, y_pred_test)
f1_test = f1_score(y_test, y_pred_test)
accuracy_test = accuracy_score(y_test, y_pred_test)
roc_auc_test = roc_auc_score(y_test, y_proba_test)

print(f"\n✓ RÉSULTATS FINAUX (TEST)")
print("-" * 60)
print(f"Modèle:       {best_model_name}")
print(f"Seuil:        {optimal_threshold:.2f}")
print(f"\nPerformances:")
print(f"  Recall:     {recall_test:.3f} ({recall_test*100:.1f}%)")
print(f"  Precision:  {precision_test:.3f} ({precision_test*100:.1f}%)")
print(f"  F1-Score:   {f1_test:.3f}")
print(f"  Accuracy:   {accuracy_test:.3f}")
print(f"  ROC-AUC:    {roc_auc_test:.3f}")

# Matrice de confusion
cm = confusion_matrix(y_test, y_pred_test)
print(f"\nMatrice de confusion:")
print(f"  VN (Vrais Négatifs):   {cm[0,0]:,}")
print(f"  FP (Faux Positifs):    {cm[0,1]:,}")
print(f"  FN (Faux Négatifs):    {cm[1,0]:,}")
print(f"  VP (Vrais Positifs):   {cm[1,1]:,}")

# Impact métier
total_defauts = cm[1,0] + cm[1,1]
defauts_detectes = cm[1,1]
# Handle the case where no defaults are detected to avoid division by zero
taux_detection = defauts_detectes / total_defauts if total_defauts > 0 else 0

print(f"\nImpact métier:")
print(f"  Total défauts réels:     {total_defauts}")
print(f"  Défauts détectés:        {defauts_detectes} ({taux_detection*100:.1f}%)")
print(f"  Défauts manqués:         {cm[1,0]}")
print(f"  Fausses alertes:         {cm[0,1]}")
# Handle the case where no true positives are detected to avoid division by zero
ratio_fausses_vraies = cm[0,1]/defauts_detectes if defauts_detectes > 0 else float('inf')
print(f"  Ratio fausses/vraies:    {ratio_fausses_vraies:.2f}:1")


# Classification report
print(f"\nRapport de classification détaillé:")
print(classification_report(y_test, y_pred_test,
                          target_names=['Remboursement', 'Défaut']))

# Sauvegarder tout
resultats_finaux = {
    'modele': best_model_name,
    'seuil': float(optimal_threshold),
    # Hyperparameters from pre-tuned models (can be added if needed, but not from Optuna)
    'hyperparametres': None,
    'validation': {
        'recall': float(meilleur_config['recall']),
        'precision': float(meilleur_config['precision']),
        'f1': float(meilleur_config['f1'])
    },
    'test': {
        'recall': float(recall_test),
        'precision': float(precision_test),
        'f1': float(f1_test),
        'accuracy': float(accuracy_test),
        'roc_auc': float(roc_auc_test)
    },
    'confusion_matrix': cm.tolist(),
    'features': {
        'total': 118, # This might need to be updated if feature selection was applied and accepted
        'selected': X_test_final.shape[1] if isinstance(X_test_final, pd.DataFrame) else X_test_final.shape[1], # Use actual number of features
        'reduction': f"{(1 - X_test_final.shape[1]/118)*100:.1f}%" if isinstance(X_test_final, pd.DataFrame) else f"{(1 - X_test_final.shape[1]/118)*100:.1f}%"
    }
}

import json
with open('resultats_finaux_complets.json', 'w') as f:
    json.dump(resultats_finaux, f, indent=2)

print("\n✓ Résultats sauvegardés: resultats_finaux_complets.json")
print("=" * 60)


ÉVALUATION FINALE SUR ENSEMBLE DE TEST

✓ RÉSULTATS FINAUX (TEST)
------------------------------------------------------------
Modèle:       CatBoost
Seuil:        0.20

Performances:
  Recall:     0.471 (47.1%)
  Precision:  0.353 (35.3%)
  F1-Score:   0.403
  Accuracy:   0.766
  ROC-AUC:    0.740

Matrice de confusion:
  VN (Vrais Négatifs):   208
  FP (Faux Positifs):    44
  FN (Faux Négatifs):    27
  VP (Vrais Positifs):   24

Impact métier:
  Total défauts réels:     51
  Défauts détectés:        24 (47.1%)
  Défauts manqués:         27
  Fausses alertes:         44
  Ratio fausses/vraies:    1.83:1

Rapport de classification détaillé:
               precision    recall  f1-score   support

Remboursement       0.89      0.83      0.85       252
       Défaut       0.35      0.47      0.40        51

     accuracy                           0.77       303
    macro avg       0.62      0.65      0.63       303
 weighted avg       0.80      0.77      0.78       303


✓ Résultats sa

In [ ]:
# ============================================================================
# CALIBRATION DES PROBABILITÉS (OPTIONNEL)
# ============================================================================

from sklearn.calibration import CalibratedClassifierCV

print("\n" + "=" * 60)
print("CALIBRATION DES PROBABILITÉS")
print("=" * 60)

# Calibrer le meilleur modèle
calibrated_model = CalibratedClassifierCV(model_final, method='sigmoid', cv='prefit')

if meilleur_config['modele'] == 'Logistic Regression':
    calibrated_model.fit(X_val_scaled, y_val)
    y_proba_calib = calibrated_model.predict_proba(X_test_scaled)[:, 1]
else:
    calibrated_model.fit(X_val, y_val)
    y_proba_calib = calibrated_model.predict_proba(X_test_final)[:, 1]

y_pred_calib = (y_proba_calib >= meilleur_config['seuil']).astype(int)

# Comparer
f1_calib = f1_score(y_test, y_pred_calib)
recall_calib = recall_score(y_test, y_pred_calib)
precision_calib = precision_score(y_test, y_pred_calib)

print(f"\nAVANT calibration:")
print(f"  F1={f1_test:.3f}, Recall={recall_test:.3f}, Precision={precision_test:.3f}")

print(f"\nAPRÈS calibration:")
print(f"  F1={f1_calib:.3f}, Recall={recall_calib:.3f}, Precision={precision_calib:.3f}")

if f1_calib > f1_test:
    print(f"\n✓ Calibration améliore F1 de {(f1_calib-f1_test)*100:.2f}%")
else:
    print(f"\n✗ Calibration dégrade F1, on garde le modèle non calibré")


CALIBRATION DES PROBABILITÉS

AVANT calibration:
  F1=0.403, Recall=0.471, Precision=0.353

APRÈS calibration:
  F1=0.377, Recall=0.392, Precision=0.364

✗ Calibration dégrade F1, on garde le modèle non calibré


In [ ]:
# ============================================================================
# CALCUL DES MÉTRIQUES MANQUANTES SUR VALIDATION
# ============================================================================

print("\n" + "=" * 60)
print("CALCUL MÉTRIQUES COMPLÉMENTAIRES SUR VALIDATION")
print("=" * 60)

# Retrieve the final chosen model and optimal threshold from previous steps
# The model object is stored in model_final from cell xPWfN9VMna8I
# The optimal threshold is stored in optimal_threshold from cell BY9ckvL8nNR2
# The best_model_name is stored in best_model_name from cell xPWfN9VMna8I

# Select the correct validation set based on the chosen model's requirements (scaled or not)
if best_model_name == 'Logistic Regression':
    X_val_final = X_val_scaled
else:
    X_val_final = X_val # RF and CatBoost used non-scaled data in the validation step

# Prédictions validation avec modèle final et seuil optimal
y_proba_val_final = model_final.predict_proba(X_val_final)[:, 1]
y_pred_val_final = (y_proba_val_final >= optimal_threshold).astype(int)


# Métriques manquantes for the FINAL selected model on validation set using optimal threshold
recall_val_final = recall_score(y_val, y_pred_val_final)
precision_val_final = precision_score(y_val, y_pred_val_final)
f1_val_final = f1_score(y_val, y_pred_val_final)
accuracy_val_final = accuracy_score(y_val, y_pred_val_final)
roc_auc_val_final = roc_auc_score(y_val, y_proba_val_final)


print(f"\nMÉTRIQUES VALIDATION ({best_model_name} avec seuil {optimal_threshold:.2f}):")
print(f"  Recall:    {recall_val_final:.3f} ({recall_val_final*100:.1f}%)")
print(f"  Precision: {precision_val_final:.3f} ({precision_val_final*100:.1f}%)")
print(f"  F1-Score:  {f1_val_final:.3f}")
print(f"  Accuracy:  {accuracy_val_final:.3f} ({accuracy_val_final*100:.1f}%)")
print(f"  ROC-AUC:   {roc_auc_val_final:.3f}")


print(f"\nCOMPARAISON VALIDATION vs TEST:")
print(f"{'Métrique':<15} {'Validation':<12} {'Test':<12} {'Écart':<10}")
print("-" * 50)
print(f"{'F1-Score':<15} {f1_val_final:<12.3f} {f1_test:<12.3f} {abs(f1_val_final-f1_test):<10.3f}")
print(f"{'Recall':<15} {recall_val_final:<12.3f} {recall_test:<12.3f} {abs(recall_val_final-recall_test):<10.3f}")
print(f"{'Precision':<15} {precision_val_final:<12.3f} {precision_test:<12.3f} {abs(precision_val_final-precision_test):<10.3f}")
print(f"{'Accuracy':<15} {accuracy_val_final:<12.3f} {accuracy_test:<12.3f} {abs(accuracy_val_final-accuracy_test):<10.3f}")
print(f"{'ROC-AUC':<15} {roc_auc_val_final:<12.3f} {roc_auc_test:<12.3f} {abs(roc_auc_val_final-roc_auc_test):<10.3f}")

# Mise à jour du fichier JSON
# Ensure the keys exist before updating
if 'validation' not in resultats_finaux:
    resultats_finaux['validation'] = {}
if 'test' not in resultats_finaux:
     resultats_finaux['test'] = {}

resultats_finaux['validation']['recall'] = float(recall_val_final)
resultats_finaux['validation']['precision'] = float(precision_val_final)
resultats_finaux['validation']['f1'] = float(f1_val_final)
resultats_finaux['validation']['accuracy'] = float(accuracy_val_final)
resultats_finaux['validation']['roc_auc'] = float(roc_auc_val_final)

# Update test metrics from the previous cell execution
resultats_finaux['test']['recall'] = float(recall_test)
resultats_finaux['test']['precision'] = float(precision_test)
resultats_finaux['test']['f1'] = float(f1_test)
resultats_finaux['test']['accuracy'] = float(accuracy_test)
resultats_finaux['test']['roc_auc'] = float(roc_auc_test)


import json
with open('resultats_finaux_complets.json', 'w') as f:
    json.dump(resultats_finaux, f, indent=2)

print("\n✓ Fichier JSON mis à jour avec métriques validation complètes")
print("=" * 60)


CALCUL MÉTRIQUES COMPLÉMENTAIRES SUR VALIDATION

MÉTRIQUES VALIDATION (CatBoost avec seuil 0.20):
  Recall:    0.635 (63.5%)
  Precision: 0.458 (45.8%)
  F1-Score:  0.532
  Accuracy:  0.809 (80.9%)
  ROC-AUC:   0.820

COMPARAISON VALIDATION vs TEST:
Métrique        Validation   Test         Écart     
--------------------------------------------------
F1-Score        0.532        0.403        0.129     
Recall          0.635        0.471        0.164     
Precision       0.458        0.353        0.105     
Accuracy        0.809        0.766        0.043     
ROC-AUC         0.820        0.740        0.080     

✓ Fichier JSON mis à jour avec métriques validation complètes
